In [1]:
import numpy as np
import random
import time

# =========================
# 1. 参数设置
# =========================
gamma = 0.9
alpha_A = 0.1
alpha_B = 0.1
epsilon = 0.05

num_episodes = 1000
max_steps_per_episode = 100

# =========================
# 随机种子
# =========================
base_seed = int(time.time() * 1000000) % (2**32)
print(f"Base seed: {base_seed}")

run_id = 0
seed = base_seed + run_id
random.seed(seed)
np.random.seed(seed)
print(f"Seed for this run: {seed}")

# =========================
# 2. 状态与行动空间
# =========================
states = ["CC", "CD", "DC", "DD"]
num_states = len(states)

# 动作：0 = D, 1 = C
actions = [0, 1]

def action_to_char(a):
    return "C" if a == 1 else "D"

# =========================
# 3. 即时收益矩阵
# =========================
payoff_matrix = {
    ("CC", "CC"): (4, 4), ("CC", "CD"): (1, 5), ("CC", "DC"): (5, 1), ("CC", "DD"): (2, 2),
    ("CD", "CC"): (4, 4), ("CD", "CD"): (1, 5), ("CD", "DC"): (5, 1), ("CD", "DD"): (2, 2),
    ("DC", "CC"): (4, 4), ("DC", "CD"): (1, 5), ("DC", "DC"): (5, 1), ("DC", "DD"): (2, 2),
    ("DD", "CC"): (4, 4), ("DD", "CD"): (1, 5), ("DD", "DC"): (5, 1), ("DD", "DD"): (2, 2)
}

# =========================
# 4. 初始化 Q 表（warm start：用上一轮结果）
# =========================
Q_A = np.array([
    [36.37543815, 36.09320668],
    [35.91898920, 33.92022832],
    [35.97780508, 34.01886474],
    [36.25463994, 36.06546395]
], dtype=float)

Q_B = np.array([
    [36.54050933, 36.29385224],
    [36.36715929, 34.07739378],
    [36.44759073, 33.97842444],
    [37.33104129, 37.00729492]
], dtype=float)

# =========================
# 5. 当前策略提取函数
# =========================
def get_current_strategies():
    """返回当前 Q 表下的策略字典（每个状态下 argmax 动作）"""
    strategy_A = {}
    strategy_B = {}
    for s in range(num_states):
        best_A = np.argmax(Q_A[s])
        best_B = np.argmax(Q_B[s])
        strategy_A[states[s]] = action_to_char(best_A)
        strategy_B[states[s]] = action_to_char(best_B)
    return strategy_A, strategy_B

# =========================
# 6. 先输出 warm start 的 Q 与策略
# =========================
print("\n==============================")
print("初始 Q 值表（warm start）")
print("==============================")

print("\nQ_A:")
print(Q_A)

print("\nQ_B:")
print(Q_B)

strategy_A, strategy_B = get_current_strategies()
print("\n初始策略字典")
print("Agent A:", strategy_A)
print("Agent B:", strategy_B)

# =========================
# 7. Independent Q-Learning
# =========================
for episode in range(num_episodes):

    for initial_state in range(num_states):
        state = initial_state

        for step in range(max_steps_per_episode):

            # --- 玩家 A 行动（ε-greedy + 平局随机） ---
            if random.random() < epsilon:
                action_A = random.choice(actions)
            else:
                max_q = np.max(Q_A[state])
                candidates = np.where(Q_A[state] == max_q)[0]
                action_A = np.random.choice(candidates)

            # --- 玩家 B 行动（ε-greedy + 平局随机） ---
            if random.random() < epsilon:
                action_B = random.choice(actions)
            else:
                max_q = np.max(Q_B[state])
                candidates = np.where(Q_B[state] == max_q)[0]
                action_B = np.random.choice(candidates)

            # --- 即时收益 ---
            joint_action = action_to_char(action_A) + action_to_char(action_B)
            reward_A, reward_B = payoff_matrix[(states[state], joint_action)]

            # --- 状态转移 ---
            next_state = states.index(joint_action)

            # --- Q 更新 ---
            Q_A[state, action_A] += alpha_A * (
                reward_A + gamma * np.max(Q_A[next_state]) - Q_A[state, action_A]
            )
            Q_B[state, action_B] += alpha_B * (
                reward_B + gamma * np.max(Q_B[next_state]) - Q_B[state, action_B]
            )

            state = next_state

# =========================
# 8. 输出最终 Q 表与最优策略
# =========================
print("\n==============================")
print("最终 Q 值表（Agent A）")
print("==============================")
print(Q_A)

print("\n==============================")
print("最终 Q 值表（Agent B）")
print("==============================")
print(Q_B)

print("\n==============================")
print("最终最优马尔可夫策略对")
print("==============================")

strategy_A, strategy_B = get_current_strategies()

for s in range(num_states):
    print(
        f"状态 {states[s]} : "
        f"A → {strategy_A[states[s]]}, "
        f"B → {strategy_B[states[s]]}"
    )

print("\n策略字典表示（便于统计 / 写论文）")
print("Agent A:", strategy_A)
print("Agent B:", strategy_B)

# =========================
# 9. 打印训练统计
# =========================
total_updates = num_episodes * num_states * max_steps_per_episode
print("\n==============================")
print("训练统计")
print("==============================")
print(f"总Q更新次数 = {total_updates}")

Base seed: 2083980972
Seed for this run: 2083980972

初始 Q 值表（warm start）

Q_A:
[[36.37543815 36.09320668]
 [35.9189892  33.92022832]
 [35.97780508 34.01886474]
 [36.25463994 36.06546395]]

Q_B:
[[36.54050933 36.29385224]
 [36.36715929 34.07739378]
 [36.44759073 33.97842444]
 [37.33104129 37.00729492]]

初始策略字典
Agent A: {'CC': 'D', 'CD': 'D', 'DC': 'D', 'DD': 'D'}
Agent B: {'CC': 'D', 'CD': 'D', 'DC': 'D', 'DD': 'D'}

最终 Q 值表（Agent A）
[[36.88492508 38.2265183 ]
 [35.87154975 33.69979167]
 [35.98279658 33.69216597]
 [35.92131948 37.47202068]]

最终 Q 值表（Agent B）
[[36.83678358 38.69183656]
 [35.91279544 33.78044111]
 [35.90864681 33.78668836]
 [36.47937533 38.12960247]]

最终最优马尔可夫策略对
状态 CC : A → C, B → C
状态 CD : A → D, B → D
状态 DC : A → D, B → D
状态 DD : A → C, B → C

策略字典表示（便于统计 / 写论文）
Agent A: {'CC': 'C', 'CD': 'D', 'DC': 'D', 'DD': 'C'}
Agent B: {'CC': 'C', 'CD': 'D', 'DC': 'D', 'DD': 'C'}

训练统计
总Q更新次数 = 400000
